<a href="https://colab.research.google.com/github/prithwis/GeoSim/blob/main/GeoSim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/images/gsBanner_0.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

## GeoSim


## Installs, Imports ..

In [1]:
print('ॐ श्री सरस्वत्यै नमः')

ॐ श्री सरस्वत्यै नमः


In [2]:
!pip install -q python-docx
from IPython.display import Markdown, display
import json
from datetime import datetime
from zoneinfo import ZoneInfo
import pytz

In [3]:
!wget -O gsLLM.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsLLM.py
!wget -O gsWorld.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsWorld.py
!wget -O gsActors.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsActors.py
!wget -O gsZeitWorld.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsZeitWorld.py
!wget -O gsSpine.py -q https://raw.githubusercontent.com/prithwis/GeoSim/refs/heads/main/gsUtils/gsSpine.py

from gsLLM import *
from gsWorld import *
from gsActors import *
from gsSpine import *
from gsZeitWorld import *

In [4]:
import importlib

import gsLLM
import gsWorld
import gsActors
import gsSpine
import gsZeitWorld


importlib.reload(gsLLM)
importlib.reload(gsWorld)
importlib.reload(gsActors)
importlib.reload(gsSpine)
importlib.reload(gsZeitWorld)


from gsLLM import *
from gsWorld import *
from gsActors import *
from gsSpine import *
from gsZeitWorld import *


In [5]:
# ------------------------------------------------
# Use OpenAI
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
model = "gpt-6-luna"
#model = "gpt-6.1-sol"
#model = "gpt-6-astra"

# ------------------------------------------------
# Use Claude
# ------------------------------------------------
#!pip install -q anthropic

#provider = "claude"
#model = "claude-haiku-5-5"
#model = "claude-sonnet-5-5"

# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

# ------------------------------------------------
# Connect to Google Drive
# ------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF
Mounted at /content/drive


In [6]:
# ------------------------------------------------
# Locate and load all domain informatio
# ------------------------------------------------

DOMAIN = 'Silicon'

GeoSim_HOME = "/content/drive/MyDrive/Prithwis/GeoSim/"+DOMAIN
print(GeoSim_HOME)

#!ls -al "{GeoSim_HOME}"

WORLD_DESC = Path(GeoSim_HOME) / "WorldDescription"
ROLE_DIR = Path(GeoSim_HOME) / "Roles"
MD_DIR = Path(GeoSim_HOME) / "WorldFacts/Markdown"

eventFile = Path(GeoSim_HOME) / "WorldDescription" / "TriggerEvents_V1.json"

# Read actor definitions
actorFile = Path(GeoSim_HOME) / "WorldDescription" / "Actors_V1.json"

# ------------------------------------------------------------
# Load ZeitWorld role and prompts
# ------------------------------------------------------------

#ROLE_DIR = Path(WORLDFACTS).parent / "Roles"

ZW_Role = (ROLE_DIR / "ZW_Role_v0.txt").read_text(encoding="utf-8").strip()

ZW_Initialisation_Prompt = (
    ROLE_DIR / "ZW_Initialisation_Prompt_v1.txt"
).read_text(encoding="utf-8").strip()

ZW_Update_Prompt = (
    ROLE_DIR / "ZW_Update_Prompt_v0.txt"
).read_text(encoding="utf-8").strip()


AC_Generic = (ROLE_DIR / "AC_Role_Generic_v0.txt").read_text(encoding="utf-8").strip()
AC_Strategy = (ROLE_DIR / "AC_Strategy_Flex2_v0.txt").read_text(encoding="utf-8").strip()

/content/drive/MyDrive/Prithwis/GeoSim/Silicon


## Defining the World

In [7]:
# Describe the World
#
world_facts = getWorldFacts(MD_DIR)
worldVariables = loadWorldVariables(WORLD_DESC / "WorldVariables_V1.json")

# Identify Actors

ActorL, ActorD = initialiseActors(
    actorFile, client, model, provider, effort="medium"
)


Loaded 5 WorldFacts files
Total characters: 1,074,557


In [8]:
print("Actors loaded:", len(ActorD))
for actor_id, actor in ActorD.items():
    print(actor_id, ":", actor.name)

print("Actors loaded:", len(ActorL))
for actor in ActorL:
    print(actor.actor_id, ":", actor.name)

Actors loaded: 2
US : United States
CN : China
Actors loaded: 2
US : United States
CN : China


In [9]:
ActorD["CN"].show()

Actor ID    : CN
Name        : China
Description : Chinese government and its semiconductor and AI technology policy apparatus.
Type        : LLM


In [10]:
# Initialise event history
#

E0 = getTriggerEvent(eventFile, 0)  # 0 is US action, 1 is CN action
E0.event_id = "E0"
E = []
E.append(E0)
E[0].show()


Event : E0
-------------------------------------------------------
Actor     : US
Action    : The United States announces substantially tighter export restrictions on advanced AI accelerators and semiconductor manufacturing equipment destined for China, with expanded licensing requirements and enforcement.
Rationale : Limit China's access to strategic semiconductor technologies and advanced AI computing capabilities.
selection_weight : None


### Zeitworld

In [11]:
# ------------------------------------------------------------
# Instantiate ZeitWorld Agent
# ------------------------------------------------------------

ZW = ZeitWorld(
    client=client,
    role=ZW_Role,
    model=model,
    provider=provider,
    effort="medium"
)

print("ZeitWorld agent created; WorldState not yet initialised.")

# ------------------------------------------------------------
# Initialise World State
# ------------------------------------------------------------


W0 = ZW.initialise(
    worldVariables,
    world_facts,
    ZW_Initialisation_Prompt
)
W = []
W.append(W0)

print("WorldState initialised:", ZW.initialised)

# ------------------------------------------------------------
# Update World State after Trigger Event
# ------------------------------------------------------------

W_new = ZW.update(W, E[0], ZW_Update_Prompt)
W.append(W_new)

print("WorldStates:", len(W))


ZeitWorld agent created; WorldState not yet initialised.
OPENAI | gpt-6-luna | Effort: medium | In: 223,104 | Cached: 0 | Out: 2,395 | Reasoning: 756 | Time: 26.0s
WorldState initialised: True
ZeitWorld Updating WorldState ---------- W0
OPENAI | gpt-6-luna | Effort: medium | In: 2,977 | Cached: 0 | Out: 2,393 | Reasoning: 461 | Time: 17.3s
Generated WorldState ---------- W1
WorldStates: 2


In [12]:
W[0].show()
W[1].show()
E[0].show()



World State : W0
-------------------------------------------------------
 0  Lithography Access                 100.00
 1  Fabrication Capacity               100.00
 2  Packaging Capacity                 100.00
 3  HBM Availability                   100.00
 4  Compute Access                     100.00
 5  Export Restrictions                100.00
 6  Taiwan Stability                   100.00
 7  Supply Resilience                  100.00

World State : W1
-------------------------------------------------------
 0  Lithography Access                  97.00
 1  Fabrication Capacity                98.00
 2  Packaging Capacity                 100.00
 3  HBM Availability                   100.00
 4  Compute Access                      96.00
 5  Export Restrictions                110.00
 6  Taiwan Stability                   100.00
 7  Supply Resilience                   98.00

Event : E0
-------------------------------------------------------
Actor     : US
Action    : The United States ann

# Actors

In [13]:
def nextActor():
    last_id = E[-1].actor

    last_index = ActorL.index(ActorD[last_id])
    next_index = (last_index + 1) % len(ActorL)

    next_actor = ActorL[next_index]
    return next_actor


In [14]:
options = nextActor().propose( W, E, AC_Generic, AC_Strategy)
for option in options:
    option.show()

OPENAI | gpt-6-luna | Effort: medium | In: 2,064 | Cached: 0 | Out: 317 | Reasoning: 134 | Time: 5.4s

Event : 
-------------------------------------------------------
Actor     : CN
Action    : Expand state-backed investment and procurement for domestic semiconductor equipment, advanced packaging, AI accelerators, and other substitute technologies, coordinating firms and research institutions around priorities less dependent on restricted imports.
Rationale : The tighter controls increase the risk of relying on foreign equipment and accelerators. Building domestic capabilities could reduce that exposure, although access to EUV remains constrained and advanced fabrication depends on a broader international supplier network.
selection_weight : None

Event : 
-------------------------------------------------------
Actor     : CN
Action    : Open a diplomatic and commercial negotiation with the United States and relevant trading partners seeking narrower licensing restrictions or case-by-

### Chanakya

In [ ]:
#!ls -l "{CH_PATH}"

In [ ]:
# Load Chanakya roles and strategy
#
# Generic Role : defines the common strategic discipline of all Chanakyas
# US Role      : defines US objectives, interests and constraints
# Iran Role    : defines Iranian objectives, interests and constraints
# BinaryHD     : generates two contrasting Hawk and Dove candidates
# ============================================================

CH_Generic_Role = getPrompt(f"{CH_PATH}/CH_Role_Generic_v0.txt")
CH_US_Role      = getPrompt(f"{CH_PATH}/CH_Role_US_v0.txt")
CH_IR_Role      = getPrompt(f"{CH_PATH}/CH_Role_Iran_v0.txt")

#CH_Strategy     = getPrompt(f"{CH_PATH}/CH_Strategy_BinaryHD_v0.txt")
#CH_Strategy     = getPrompt(f"{CH_PATH}/CH_Strategy_Flex2_v261009.txt")
CH_Strategy     = getPrompt(f"{CH_PATH}/CH_Strategy_Flex2_v261009_A.txt")

# Instantiate Chanakya agents
#
# Each Chanakya represents a national actor with its own
# interests and constraints, using the current Binary Hawk/Dove strategy.
# The registry maps each actor to its corresponding Chanakya.
# ============================================================

Chanakya_US = Chanakya_Agent(
    agent_id="ChanakyaUS_HD",
    actor="US",
    generic_role=CH_Generic_Role,
    specific_role=CH_US_Role,
    strategy=CH_Strategy
)

Chanakya_US.show()

Chanakya_IR = Chanakya_Agent(
    agent_id="ChanakyaIR_HD",
    actor="Iran",
    generic_role=CH_Generic_Role,
    specific_role=CH_IR_Role,
    strategy=CH_Strategy
)

Chanakya_IR.show()

ChanakyaRegistry = {
    "US": Chanakya_US,
    "Iran": Chanakya_IR
}

Chanakya : ChanakyaUS_HD
Actor    : US
Chanakya : ChanakyaIR_HD
Actor    : Iran


### Centaur

In [ ]:
# Load Centaur roles
#
# Generic Role : defines the common adjudication discipline of all Centaurs
# UNSG Role    : provides a fair, independent UN diplomatic perspective
# ============================================================

CT_Generic_Role = getPrompt(f"{CT_PATH}/CT_Generic_Role_v0.txt")
CT_UNSG_Role      = getPrompt(f"{CT_PATH}/CT_UNSG_Role_v0.txt")

# Instantiate Centaur adjudicator
#
# CentaurUNSG evaluates Chanakya's Candidate Events from the
# perspective of a fair and independent UN Secretary-General.
# ============================================================
Centaur_UNSG = Centaur_Agent(
    agent_id="CentaurUNSG",
    generic_role=CT_Generic_Role,
    adjudication_role=CT_UNSG_Role
)

### Reporter

In [ ]:
# ============================================================
# Load PressReporter Roles and Prompts
#
# Generic Role defines reporting discipline.
# Narrative Prompt produces the human-readable Scenario history.
# Structured Prompt produces the machine-oriented Markdown record.
# ============================================================

PR_Generic_Role = getPrompt(f"{PR_PATH}/PR_Generic_Role_v0.txt")

#PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_v0.txt")
#PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_Brief_v2.txt")
PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_Brief_v3.txt")

PR_Structured   = getPrompt(f"{PR_PATH}/PR_Structured_v0.txt")


# ============================================================
# Instantiate Human Reporter
# ============================================================

Reporter = Reporter_Agent(
    client=client,
    model=model,
    agent_id="Vyasa",
    generic_role=PR_Generic_Role
)

## Initialisation of Scenario

In [ ]:
# Initialise Simulation State
#
# W stores the evolving WorldStates of the simulation.
# E stores the sequence of Events defining the Scenario.
# W0_Template defines the empty structure from which W0 is created.
# ------------------------------------------------------------

E = []
W = []

W0_Template = WorldState(
    state_id="W0",
    variables=worldVariables
)

# Initialise the first WorldState
#
# ZeitWorld uses the WorldFactBase and W0 template to create W0.
# The completed WorldState is automatically appended to W as W[0].
# ============================================================

_ = ZeitWorld.initialise(W,W0_Template, WorldFactBase, ZW_Initialisation_Prompt)

#print(ZeitWorld.initialised)

Initialising WorldState ... 
Model: gpt-5.6-luna | Effort: medium | Time: 23.3s | Input: 182,289 | Cached: 0 | Output: 1,879 | Reasoning: 224 | Visible: 1,655 | Total: 184,168 | Speed: 80.5 tok/s | Status: completed


In [ ]:
# A first look at the intial World State W[0]
# The initial values are all set to 100, as the baseline
#

#W[0].show(-2)  # ---- just the world vectors
#W[0].show()    # ---- the entire state
#W[0].show(4)    # ---- details of any one vector

In [ ]:
# Define initial Trigger options
#
# Two alternative Events are available to start the simulation.
# One is manually selected as E0; thereafter Events are generated
# and selected by the simulation.
# ============================================================

Trigger1 = Event(
    actor="Iran",
    action=(
        "Iran declares the Strait of Hormuz closed to all commercial shipping, "
        "orders its forces to prevent tanker transit, and launches attacks against "
        "US naval forces operating in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


Trigger2 = Event(
    actor="US",
    action=(
        "The United States announces an immediate suspension of offensive military "
        "operations against Iran and offers direct negotiations on maritime security "
        "and sanctions relief, while maintaining defensive protection of US forces "
        "and commercial shipping in the Gulf."
    ),
    rationale = "Initial Trigger for US"
)

# ============================================================================

Triggers = {
    "TR1": Trigger1,
    "TR2": Trigger2
}

In [ ]:
# Select the initial Trigger
#
# The manually selected DefaultTrigger starts the first epoch.
# Its selection weight is None because it was not chosen by Shakuni.
# ============================================================
#DefaultTrigger = Trigger1
#DefaultTrigger = Trigger2
#DefaultTrigger.show()
#print(DefaultTrigger.selection_weight)


In [ ]:
# Initialise / Reset Scenario
#
# Retains W0 and clears all subsequent WorldStates and Events,
# restoring the simulation to its starting condition.
# ============================================================

def resetWorld():
    del W[1:]
    E.clear()

    print("Scenario reset to W0")

# ------------------------------------------------------------

resetWorld()

Scenario reset to W0


### Define Single Epoch

In [ ]:
#-------------------------------------------------
#
# An Event and the resultant WorldState is an Epoch
#
#-------------------------------------------------
def runEpoch():
    # ============================================================
    # EPOCH STEP 1 — Establish Current Trigger
    #
    # The epoch is derived from the current WorldState history.
    # For the first epoch, DefaultTrigger becomes E0.
    # Thereafter, the last Reaction becomes the next Trigger.
    # ============================================================

    epoch = len(W) - 1
    if len(E) == 0:
        DefaultTrigger.event_id = f"E{epoch}"
        E.append(DefaultTrigger)
    else:
        assert E[epoch] is E[-1]

    #E[epoch].show()

    #print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 1 complete" )
    #print("Epoch ",epoch, " Starts ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

    # ============================================================
    # EPOCH STEP 2 — Update WorldState
    #
    # ZeitWorld applies Trigger E[epoch] to W[epoch] and creates
    # the next WorldState W[epoch+1].
    # ============================================================

    _ = ZeitWorld.update( W,E[epoch],ZW_Update_Prompt)

    #E[epoch].show()
    #W[epoch+1].show(-2)

    #print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 2 complete" )

    # ============================================================
    # EPOCH STEP 3 — Select Active Actor
    #
    # The Turn Controller identifies which actor responds to the
    # Current Trigger and activates its corresponding Chanakya.
    # ============================================================

    #ActiveChanakya = getActiveChanakya(E[-1], ChanakyaRegistry)
    ActiveChanakya = getActiveChanakya(E[epoch], ChanakyaRegistry)
    #ActiveChanakya.show()

    #print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 3 complete" )

    # ============================================================
    # EPOCH STEP 4 — Generate Candidate Reactions
    #
    # The Active Chanakya evaluates the updated WorldState and
    # generates Candidate Events as possible reactions to the Trigger.
    # ============================================================

    Candidates = ActiveChanakya.propose(
        client=client,
        model=model,
        trigger=E[epoch],
        world=W[epoch+1]
    )

    #for i, candidate in enumerate(Candidates):
    #    print(f"\nCandidate Reaction {i+1}")
    #    print("-" * 60)
    #    candidate.show()

    #print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 4 complete" )

    # ============================================================
    # EPOCH STEP 5 — Adjudicate Candidate Reactions
    #
    # Centaur evaluates the Candidate Events against the current
    # WorldState and assigns each a relative simulation weight.
    # ============================================================

    weights = Centaur_UNSG.evaluate(
        client,
        model,
        E[epoch],
        W[epoch + 1],
        Candidates
    )

    #print("\nCentaur Weights")
    #print("----------------")
    #print("Weights",weights)

    #print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 5 complete" )

    # ============================================================
    #
    # Centaur weights represent the relative conditional likelihood
    # of each Candidate at this decision point.
    #
    # Shakuni makes a single stochastic selection using these weights.
    # The selected Candidate becomes the Reaction, retains its Centaur
    # weight as selection_weight, receives the next Event ID, and is
    # appended to E as the Trigger for the next epoch.
    # ============================================================

    Reaction = Shakuni(Candidates, weights)

    Reaction.event_id = f"E{len(E)}"
    E.append(Reaction)

    #print("\nShakuni has spoken")
    #print("------------------")
    #Reaction.show()
    #print(Reaction.selection_weight)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E) )
    print("Epoch ends .............................................")



##Scenario | Multiple Epochs

In [ ]:
#-------------------------------------------------
#
# Collection of Epochs is a Scenario
#
#-------------------------------------------------

def runScenario(MaxEvents = 3,_scenarioID = "dummy", RunSuffix = "000000-000000"):

    #print("Scenario ",_scenarioID, " starts ----------------------------")

    resetWorld()
    while len(E) < MaxEvents:
        runEpoch()

    # Apply the final Event and close the Scenario
    _ = ZeitWorld.update(W, E[-1], ZW_Update_Prompt)

    # Structural integrity check
    assert len(E) == MaxEvents
    assert len(W) == MaxEvents + 1
    assert len(W) == len(E) + 1

    print("Scenario ", _scenarioID," complete with ", "Events :", len(E), "WorldStates :", len(W))

    #print("Now Generating Report for scenario ",_scenarioID )


    # ============================================================
    # Generate Narrative Scenario Report
    #
    # PressReporter converts the completed Scenario (W + E) into a
    # human-readable narrative using the Narrative reporting prompt.
    # The resulting report is then formatted and saved as a Word document.
    # ============================================================
    '''
    NarrativeReport = Reporter.GenerateReport( W,E, PR_Narrative)
    ReportTitle="Middle East Oil Security — Scenario Brief"
    NarrativeFile = createNarrativeDocx( NarrativeReport, W, E, _scenarioID,ReportTitle,ReportSuffix )
    print(f"Narrative Scenario Report written to: {NarrativeFile}")
    '''
    # ============================================================
    # Generate Structured Scenario Report
    #
    # Produces a structured Markdown representation of the completed
    # Scenario for subsequent machine-based analysis.
    #
    # ============================================================

    #StructuredReport = Reporter.GenerateReport( W, E, PR_Structured)
    StructuredReport = GenerateStructuredReport(W, E)
    StructuredFile = f"MOS_{_scenarioID}_{ReportSuffix}.md"

    with open(StructuredFile, "w", encoding="utf-8") as f:
        f.write(StructuredReport)

    print(f"Structured Scenario Report written to: {StructuredFile}")
    print("Scenario ends ------------------------------------------------------------")
    #display(Markdown(StructuredReport))



##Ensemble | Multiple Scenarios

In [ ]:
#-------------------------------------------------
#
# Collection of Scenarios is an Ensemble
#
#-------------------------------------------------

# Select the initial Trigger from the registry
# ============================================================
TriggerID = "TR2"
DefaultTrigger = Triggers[TriggerID]
#DefaultTrigger.show()

#
MaxEvents = 3
NumOfRuns = 2
StartRun = 1
EndRun = StartRun + NumOfRuns - 1


ScenarioID = f"{TriggerID}ME{MaxEvents}"

for run in range(StartRun, EndRun + 1):
    print("Run # ",run, '===================================================')
    now = datetime.now(ZoneInfo("Asia/Kolkata"))
    ReportSuffix = now.strftime('%y%m%d_%H%M%S')
    runScenario(
        MaxEvents=MaxEvents,
        _scenarioID=f"{ScenarioID}_R{run:02d}",
        RunSuffix = ReportSuffix
    )

Run #  21 ===================================================
Scenario reset to W0
ZeitWorld Updating WorldState ---------- W0
Model: gpt-5.6-luna | Effort: medium | Time: 22.0s | Input: 2,975 | Cached: 0 | Output: 2,224 | Reasoning: 250 | Visible: 1,974 | Total: 5,199 | Speed: 101.0 tok/s | Status: completed
Appended  WorldState ---------- W1
Chanakya working ---------- ChanakyaIR_HD
Model: gpt-5.6-luna | Effort: medium | Time: 5.5s | Input: 2,232 | Cached: 0 | Output: 344 | Reasoning: 142 | Visible: 202 | Total: 2,576 | Speed: 62.7 tok/s | Status: completed
Centaur working ---------- CentaurUNSG
Model: gpt-5.6-luna | Effort: medium | Time: 2.0s | Input: 1,990 | Cached: 0 | Output: 115 | Reasoning: 99 | Visible: 16 | Total: 2,105 | Speed: 57.5 tok/s | Status: completed
epoch : 0 len(W): 2 len(E): 2
Epoch ends .............................................
ZeitWorld Updating WorldState ---------- W1
Model: gpt-5.6-luna | Effort: medium | Time: 21.4s | Input: 3,295 | Cached: 0 | Output: 

In [ ]:
# 1. Scenario identifier
print(ScenarioID)

# 2. Display destination
print("Drive Reports Directory:", OP_PATH)

# 3. Show existing files and folders
!ls -lh "{OP_PATH}"

# 4. Create timestamped directory using IST
NEW_DIR = f"{OP_PATH}/{ScenarioID}_$(TZ=Asia/Kolkata date +%y%m%d_%H%M%S)"
!mkdir -p "{NEW_DIR}"

# 5. Copy MOS reports from /content
!cp -v /content/MOS_TR*.* "{NEW_DIR}/"

# 6. Verify copied files
!ls -lh "{NEW_DIR}"

TR2ME3
Drive Reports Directory: /content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS//Reports
total 20K
drwx------ 2 root root 4.0K Oct 10 23:23 TR1ME3_261009_172242-Ensemble-20
drwx------ 2 root root 4.0K Oct 12 06:52 TR1ME3_261012_122221
drwx------ 2 root root 4.0K Oct 10 23:23 TR2ME3_261009_192040-Ensemble-20
drwx------ 2 root root 4.0K Oct 11 00:00 TR2ME3_261011_053006
drwx------ 2 root root 4.0K Oct 11 02:10 TR2ME3_261011_074049
'/content/MOS_TR2ME3_R21_261012_123014.md' -> '/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS//Reports/TR2ME3_261012_125350/MOS_TR2ME3_R21_261012_123014.md'
'/content/MOS_TR2ME3_R22_261012_123134.md' -> '/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS//Reports/TR2ME3_261012_125350/MOS_TR2ME3_R22_261012_123134.md'
'/content/MOS_TR2ME3_R23_261012_123304.md' -> '/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS//Reports/TR2ME3_261012_125350/MOS_TR2ME3_R23_261012_123304.md'
'/content

In [ ]:
# 3. Show existing files and folders
!ls -lh "{OP_PATH}"

total 24K
drwx------ 2 root root 4.0K Oct 10 23:23 TR1ME3_261009_172242-Ensemble-20
drwx------ 2 root root 4.0K Oct 12 06:52 TR1ME3_261012_122221
drwx------ 2 root root 4.0K Oct 10 23:23 TR2ME3_261009_192040-Ensemble-20
drwx------ 2 root root 4.0K Oct 11 00:00 TR2ME3_261011_053006
drwx------ 2 root root 4.0K Oct 11 02:10 TR2ME3_261011_074049
drwx------ 2 root root 4.0K Oct 12 07:23 TR2ME3_261012_125350


To see a single scenario in a human-readable docx format, please use the Notebook MOS_MD2DOCX

### Debugger

In [ ]:
#print(E)

#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)